### Transformer-from-Scratch

A clean Pytorch re-implementation of the Transformer architecture described in Attention is All You Need

#### Transformer 模型代码拆解
1. Positional Encoding
2. Multi-Head Attention
3. Feed Forward Network
4. Transformer Encoder Layer
5. Transformer Encoder
6. Transformer Decoder Layer
7. Transformer Decoder
8. Transformer Model
9. mask function
10. Example usage

整体框架：
<img src="transformer.jpg" height="320">

- Input / Output Embedding：把 token id 映射到连续向量。
- Positional Encoding：补位置信息，不补语义内容。
- Encoder：负责编码输入序列，产出 `memory`。
- Decoder：负责根据目标前缀 + `memory` 生成当前表示。
- Generator：把最后的 hidden state 投影到词表 logits。

Self-Attention：同一序列内部彼此看，建模 token 和 token 的关系。

Masked Self-Attention：只能看左边，防止训练时偷看未来。

Cross-Attention：decoder 用当前状态去查 encoder 的 `memory`。

FFN：每个位置各自过一个两层 MLP，做特征加工。

Add + Norm：保留原输入通路，同时稳定训练。

Causal Mask：本质是上三角屏蔽，防信息泄漏。

## Transformer 面经题整理

### 1 分钟讲清 Transformer

**面试短答：**

Transformer 是一种 **基于注意力、去掉循环和卷积** 的序列建模架构。标准版是 **encoder-decoder** 结构：encoder 每层包含 **Multi-Head Self-Attention + FFN**，decoder 每层包含 **Masked Self-Attention + Cross-Attention + FFN**，并配合 **残差连接和归一化**。它相对 RNN 的核心优势是：**并行性更强、长距离依赖路径更短**；核心代价是标准 self-attention 对序列长度通常有 **二次复杂度**。今天很多大模型仍然是 Transformer 变体，只是把位置编码、归一化和 attention 实现做得更强更快，比如 **RoPE、Pre-LN、RMSNorm、FlashAttention**。

**展开回答：**

如果从代码角度看，你当前工程里已经把标准 Transformer 主干拆成了几块：

- `MHA.py`：注意力主体，核心公式是 `softmax(QK^T / sqrt(d_k))V`
- `FFN.py`：逐位置的两层 MLP，负责特征加工
- `Encoder.py`：堆叠 self-attention + FFN
- `Decoder.py`：堆叠 masked self-attention + cross-attention + FFN
- `Transformer.py`：embedding -> encoder -> decoder -> generator 的完整拼装

因此面试时如果被问“Transformer 靠什么有表达能力”，标准回答不是只说 attention，而是说 **attention + FFN + 残差 + 归一化 + 位置建模** 一起构成了完整系统。

### 1）为什么 Transformer 比 RNN 更适合大规模训练？

**面试短答：**

因为 RNN 的时间步存在严格串行依赖，而 Transformer 的 self-attention 可以在整个序列上并行计算，所以更容易吃满 GPU/TPU 这类矩阵计算硬件。同时，Transformer 的长距离依赖路径更短，两位置之间一次 attention 就能直接交互，而 RNN 需要跨很多时间步传递。

**展开回答：**

RNN 在第 `t` 步的 hidden state 依赖第 `t-1` 步，因此训练时即使使用 BPTT，也很难彻底并行。Transformer 则不同，给定输入张量

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

Q、K、V 可以对所有位置一起做线性映射，然后一次性完成矩阵乘法：

$$
QK^T \in \mathbb{R}^{B \times h \times L \times L}
$$

所以训练时更像大规模矩阵运算，而不是时间步 for-loop。与此同时，RNN 中位置 `i` 和位置 `j` 的依赖路径长度大约是 `|i-j|`，而 self-attention 中两者可以直接交互，依赖路径可视为 `O(1)`。这也是为什么它更适合大规模预训练。

**但要补一句：** 并行性更强不等于没有代价。标准 attention 的 `L x L` 交互会让长序列在显存和算力上很贵，这是 Transformer 的核心瓶颈之一。

### 2）Self-Attention 的公式是什么？每一项是什么意思？

**面试短答：**

标准 self-attention 公式是：

$$
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

其中 `Q` 表示“我现在想找什么”，`K` 表示“我这里有什么信息值得你关注”，`V` 表示“如果你关注我，你最终拿走什么内容”。

**展开回答：**

设输入为

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

通过三组线性映射得到：

$$
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

如果有 `h` 个头，通常 reshape 成：

$$
Q,K,V \in \mathbb{R}^{B \times h \times L \times d_k},\quad d_k = d_{model}/h
$$

然后：

1. `QK^T`：计算每个 query 对每个 key 的相关性分数
2. `/ sqrt(d_k)`：控制分数尺度
3. `softmax`：把分数转成注意力权重
4. `@ V`：根据权重对 value 做加权求和，得到新的表示

和你当前 `MHA.py` 里的实现是一一对应的：

```python
scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
attn_weights = torch.softmax(scores, dim=-1)
output = torch.matmul(attn_weights, V)
```

<img src="Scaled_Dot_Product_Attention.png" height="280">

### 3）为什么要除以 $\sqrt{d_k}$

**面试短答：**

因为如果不缩放，`QK^T` 的数值方差会随着维度 `d_k` 增大而变大，softmax 很容易过饱和，导致梯度变差。除以 `sqrt(d_k)` 后，attention score 的尺度更稳定。

**展开回答：**

设某一对 query、key 的每个维度近似独立，且满足：

$$
\mathbb{E}[q_i] = \mathbb{E}[k_i] = 0, \quad \operatorname{Var}(q_i)=\operatorname{Var}(k_i)=1
$$

则点积

$$
q \cdot k = \sum_{i=1}^{d_k} q_i k_i
$$

的方差近似满足：

$$
\operatorname{Var}(q \cdot k) \approx d_k
$$

于是缩放后：

$$
\operatorname{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) \approx 1
$$

这会带来两个直接好处：

- softmax 输入不会因为维度变大而越来越极端
- 梯度不会因为过早饱和而变小

这也是你在 `MHA.py` 里那句 `/ math.sqrt(self.d_k)` 的理论来源。注意，这和 `Transformer.py` 里 embedding 乘 `sqrt(d_model)` 不是一回事：前者是 **score 稳定化**，后者更像 **输入尺度设计**。

### 4）Multi-Head Attention 为什么比单头更好？

**面试短答：**

多头的好处是让模型能在不同子空间里并行关注不同类型的关系，而不是把所有关系都压到一个 attention 模式里。最终每个 head 的结果再拼接并投影回 `d_model`，表达能力更丰富。

**展开回答：**

单头 attention 只能给出一套权重模式。如果当前序列里同时存在：

- 语法依赖
- 指代关系
- 局部短语搭配
- 长距离主题关联

单头需要在一套分数里兼顾这些结构；多头则可以让不同头分别学不同模式。数学上，如果

$$
Q_i, K_i, V_i \in \mathbb{R}^{B \times L \times d_k}
$$

则每个头独立算：

$$
head_i = \operatorname{Attention}(Q_i, K_i, V_i)
$$

最后：

$$
\operatorname{MultiHead}(Q,K,V)=\operatorname{Concat}(head_1,\dots,head_h)W_O
$$

这和你 `MHA.py` 里的 `_split_heads()`、每头 attention、再 `_merge_heads()` + `W_o` 是一致的。

<img src="Multi_Head_Attention.png" height="300">

**但要严谨地补一句：** 多头并不保证每个头一定学到完全不同的功能，实际训练中会有头冗余、头相似的现象；只是给了模型这种能力和空间。

### 5）Encoder 和 Decoder 有什么区别？

**面试短答：**

Encoder 负责把输入序列编码成上下文化表示；Decoder 负责在已生成前缀和 encoder 输出的条件下生成目标序列。结构上，encoder 每层只有 self-attention + FFN；decoder 每层有 masked self-attention + cross-attention + FFN。

**展开回答：**

在你的实现里：

- `EncoderLayer`：`self_attn -> FFN`
- `DecoderLayer`：`masked self_attn -> cross_attn -> FFN`

区别主要体现在两点：

1. **decoder 自注意力要加 causal mask**
   因为它做自回归生成，当前位置不能偷看未来 token。

2. **decoder 多了 cross-attention**
   其中 query 来自 decoder 当前状态 `x`，key/value 来自 encoder 输出 `memory`。也就是：

$$
Q = xW_Q, \quad K = memory W_K, \quad V = memory W_V
$$

因此你可以把 encoder 理解成“读输入”，把 decoder 理解成“边看已生成前缀边查输入资料继续写”。如果去掉 encoder 和 cross-attention，结构就更接近 GPT 这类 decoder-only 模型。

### 6）为什么 Transformer 需要位置编码？

**面试短答：**

因为 self-attention 本身只看 token 间内容相关性，不天然知道顺序。如果没有位置信息，模型很难区分“我喜欢苹果”和“苹果喜欢我”这种词一样但顺序不同的句子。

**展开回答：**

设输入 token embedding 为：

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

如果直接对 `X` 做 self-attention，那么模型看到的是一组向量之间的相似性关系，它不知道“第几个位置”这个信息。因此原始 Transformer 在输入端加入位置编码：

$$
X' = X + PE
$$

其中 `PE` 在你当前 `PostionalEncoding.py` 里是固定的正弦位置编码。它的直觉是：

- 给每个位置一个唯一但连续的向量表示
- 让模型能推断绝对位置和相对位移

**面试时最好补一句：** mask 和位置编码不是一回事。mask 决定“能不能看”，位置编码决定“看到的内容处在什么顺序/位置”。

### 7）RoPE 是什么？相比绝对位置编码有什么优点？

**面试短答：**

RoPE（Rotary Positional Embedding）是一种把位置信息通过旋转直接注入到 `Q/K` 向量中的方法。它相比绝对位置编码的一个核心优点是：attention 分数天然带有相对位置信息，而且通常对长序列外推更友好。

**展开回答：**

正弦绝对位置编码的做法是 `X + PE`；RoPE 则是对 `Q` 和 `K` 的二维分量对做旋转。对某一对维度，可以写成：

$$
\begin{bmatrix}
q_{2i}' \\
q_{2i+1}'
\end{bmatrix}
=
\begin{bmatrix}
\cos \theta_{p,i} & -\sin \theta_{p,i} \\
\sin \theta_{p,i} & \cos \theta_{p,i}
\end{bmatrix}
\begin{bmatrix}
q_{2i} \\
q_{2i+1}
\end{bmatrix}
$$

对 `K` 也做同样处理。RoPE 的漂亮性质是，旋转后 `QK^T` 的结果和相对位置差有关，因此更适合建模“相隔多少”的关系。

**相比绝对位置编码的优点：**

- 相对位置信息融入 attention 分数本身
- 在很多长上下文场景里更稳
- 不需要把一个绝对位置向量直接加到 embedding 上

**工程上常见结论：** 原始正弦 PE 很适合教学和论文主干理解；RoPE 更像现代 LLM 的主流位置建模方案。

### 8）FFN 在 Transformer 里是干什么的？

**面试短答：**

FFN 负责对每个位置的表示做逐位置、非线性的特征加工。attention 负责 token 之间的信息交互，FFN 负责每个 token 自己在特征维上的深加工。

**展开回答：**

标准 FFN 一般写成：

$$
\operatorname{FFN}(x)=W_2\sigma(W_1x+b_1)+b_2
$$

在你的 `FFN.py` 里，就是：

```python
x = self.fc1(x)
x = self.activation(x)
x = self.dropout(x)
x = self.fc2(x)
```

如果输入 shape 是：

$$
(B, L, d_{model})
$$

那么 `nn.Linear` 默认作用在最后一维上，因此 FFN 是 **position-wise** 的：它不会混不同 token 的位置，只对每个 token 的特征向量做变换。

为什么常见设计是 `d_model -> d_ff -> d_model`？因为先升维可以给非线性变换一个更大的特征空间，再压回原维度以便残差连接和后续层继续处理。

### 9）为什么要残差连接和归一化？

**面试短答：**

残差连接是为了保留原信息并改善梯度传播，归一化是为了稳定数值尺度和优化过程。attention/FFN 提供表达能力，Add + Norm 提供可训练性。

**展开回答：**

标准 Post-LN 结构里常见的是：

$$
y = \operatorname{LN}(x + F(x))
$$

其中：

- `x + F(x)` 是残差连接
- `LN` 是 LayerNorm

**残差的作用：**

- 给原始输入保留一条直通路径
- 让子层学的是“修正量”而不是从头重写表示
- 深层时更利于梯度传播

**归一化的作用：**

- 稳定每层输出尺度
- 减少数值飘移
- 让更深的网络更容易训练

这就是为什么你在 `Encoder.py`、`Decoder.py` 里看到的结构不是单纯 `MHA -> FFN`，而是 `子层输出 -> Add -> Norm`。

### 10）Pre-LN 和 Post-LN 区别是什么？

**面试短答：**

Post-LN 是原始 Transformer 论文写法：`LN(x + F(x))`；Pre-LN 是先归一化再进子层：`x + F(LN(x))`。现代大模型更常用 Pre-LN，因为深层训练通常更稳定。

**展开回答：**

**Post-LN：**

$$
y = \operatorname{LN}(x + F(x))
$$

**Pre-LN：**

$$
y = x + F(\operatorname{LN}(x))
$$

两者的核心差别不是“有没有 LayerNorm”，而是 **LayerNorm 放在子层前还是残差后**。Pre-LN 通常更稳定的原因是：

- 子层输入先被规范化，梯度路径更平滑
- 深层训练时不容易出现数值不稳
- 训练大模型时更常见

但 Post-LN 的语义更直观，也更贴近原论文，所以教学代码里常常先从 Post-LN 讲起。你当前 `Encoder.py` 和 `Decoder.py` 用的就是 Post-LN 风格。

### 11）LayerNorm 和 RMSNorm 的区别？

**面试短答：**

LayerNorm 会减均值再除标准差，RMSNorm 只按均方根做缩放，不做显式去均值。RMSNorm 更简单、算子更轻，现代大模型里很常见。

**展开回答：**

**LayerNorm：**

$$
\mu = \frac{1}{d}\sum_{i=1}^d x_i, \quad
\sigma^2 = \frac{1}{d}\sum_{i=1}^d (x_i - \mu)^2
$$

$$
y_i = \gamma_i \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta_i
$$

**RMSNorm：**

$$
\operatorname{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2 + \varepsilon}
$$

$$
y_i = \gamma_i \frac{x_i}{\operatorname{RMS}(x)}
$$

所以 RMSNorm 的差别在于：

- 不做减均值
- 通常没有像 LayerNorm 那样的显式偏置项
- 更强调“只控制尺度，不控制中心”

工程上它的优点是实现简单、数值开销略低，许多现代 LLM（如一些 LLaMA 系列变体）会更偏向 RMSNorm。

### 12）Transformer 的复杂度瓶颈在哪里？

**面试短答：**

标准 Transformer 的主要瓶颈在 self-attention 的 `L x L` 交互，时间和显存开销都会随着序列长度呈二次增长。长文本时，这通常比 FFN 更容易成为瓶颈。

**展开回答：**

如果序列长度是 `L`，每头维度是 `d_k`，那么 attention 里最重的两步是：

1. `QK^T`：

$$
(L \times d_k)(d_k \times L) \rightarrow (L \times L)
$$

2. `attn_weights @ V`：

$$
(L \times L)(L \times d_k) \rightarrow (L \times d_k)
$$

因此标准 self-attention 的时间复杂度通常记作：

$$
O(L^2 d)
$$

而且中间 attention matrix 的 shape 是：

$$
(B, h, L, L)
$$

这就是显存压力最大的来源之一。严格来说，FFN 也很贵，尤其 `d_ff` 很大时计算量不小；但当序列很长时，attention 的二次项更容易成为瓶颈。面试时最好把这句说完整：

- **短序列、大宽度时，FFN 也可能很重**
- **长序列场景下，attention 的 `L^2` 更容易成为主瓶颈**

### 13）FlashAttention 是什么，为什么快？

**面试短答：**

FlashAttention 是一种 **IO-aware 的 exact attention 实现**。它不是近似 attention，而是在数学结果基本不变的前提下，通过分块计算和在线 softmax，避免把完整的 `L x L` attention matrix 频繁写回高带宽显存（HBM），所以更快也更省显存。

**展开回答：**

普通 attention 的朴素流程通常是：

1. 先算完整 `QK^T`
2. 把完整 scores 写到显存
3. 做 mask 和 softmax
4. 再把完整 weights 写到显存
5. 再做 `weights @ V`

这意味着大量中间结果在 HBM 和片上 SRAM 之间来回搬运。FlashAttention 的核心优化不是改变渐近计算量的二次项，而是减少 **内存访问成本**：

- 把 Q/K/V 分块(tile)处理
- 在块内做在线 softmax 累积
- 不显式 materialize 完整的 attention matrix 到 HBM

因此它的优势体现在：

- 更少的显存占用
- 更高的吞吐
- 更适合长序列训练和推理

**严谨一点的表述：** FlashAttention 不是把 `O(L^2)` 变成线性时间，它仍然要处理二次数量的 token-token 关系；它真正做的是 **优化 IO 路径和中间张量物化方式**，因此在现代硬件上会明显更快。

### 补充：手撕题和维度题怎么答

给定输入：

$$
X \in \mathbb{R}^{B \times L \times d_{model}}
$$

常规写法下：

$$
W_Q, W_K, W_V \in \mathbb{R}^{d_{model} \times d_{model}}
$$

所以：

$$
Q, K, V \in \mathbb{R}^{B \times L \times d_{model}}
$$

如果有 `h` 个头，则每头维度：

$$
d_k = d_{model}/h
$$

通常 reshape 成：

$$
Q, K, V \in \mathbb{R}^{B \times h \times L \times d_k}
$$

随后：

$$
QK^T \in \mathbb{R}^{B \times h \times L \times L}
$$

mask 一般加在 softmax 之前的 logits 上，对不允许关注的位置加一个极小值；decoder 的 causal mask 本质上就是为了防止训练时看见未来 token 而发生信息泄漏。

最后一个很稳的面试回答是：

> Transformer 的表达能力不是只靠 attention，而是 attention + FFN + 残差 + 归一化 + 位置建模 一起提供的。

## 再补几道常考追问题

### 14）为什么 decoder 的输入要 shifted right？

**面试短答：**

因为训练时 decoder 要学习“根据前缀预测下一个 token”。所以会把目标序列右移一位后作为输入，例如用 `<bos> 我 喜欢` 去预测 `我 喜欢 苹果`。这样当前位置只能利用过去信息，和自回归生成目标一致。

**展开回答：**

设目标序列为 `y_1, y_2, ..., y_T`，训练目标是：

$$
P(y_t \mid y_{<t}, x)
$$

因此 decoder 输入通常构造成：

- `tgt_input = [<bos>, y_1, y_2, ..., y_{T-1}]`
- `tgt_output = [y_1, y_2, ..., y_T]`

这就是 teacher forcing 的标准配对方式。

### 15）Cross-Attention 里 Q / K / V 分别来自哪里？

**面试短答：**

在 cross-attention 中，`Q` 来自 decoder 当前状态，`K` 和 `V` 来自 encoder 输出的 `memory`。也就是 decoder 决定“我要找什么”，encoder 提供“我这里有什么内容”。

**展开回答：**

如果 decoder 当前表示为 `x`，encoder 输出为 `memory`，则：

$$
Q = xW_Q, \quad K = memory W_K, \quad V = memory W_V
$$

因此 score 的 shape 是：

$$
QK^T \in \mathbb{R}^{B \times h \times T \times S}
$$

表示每个目标位置对每个源位置打分。

### 16）为什么训练时和推理时 decoder 的行为不一样？

**面试短答：**

训练时通常用 teacher forcing，一次把整段目标序列喂进去；推理时没有真值目标，只能一步一步自回归生成。两者共享同一个因果约束，所以 decoder 训练时必须加 causal mask，防止看到未来 token。

**展开回答：**

训练时，未来 token 实际上已经在张量里，如果不加 mask，当前位置就会偷看答案，造成训练和推理不一致。推理时未来 token 根本不存在，因此模型必须学会在只看前缀的条件下完成预测。

### 17）为什么现在很多大模型是 decoder-only？

**面试短答：**

因为 decoder-only 用统一的 next-token prediction 目标就能做大规模预训练，而且天然适合聊天、写作、代码生成这类生成任务。它不需要单独一套 encoder，也不需要 cross-attention，结构更统一。

**展开回答：**

encoder-decoder 更适合显式 seq2seq 任务，比如翻译、摘要；decoder-only 更适合“给前缀，继续往后写”。现代 LLM 的主流交互方式正好就是这种统一生成范式。

### 18）Embedding 和输出层有什么关系？什么是 weight tying？

**面试短答：**

输入 embedding 是把 token id 映射到向量，输出层则把 hidden state 映射回词表 logits。两者都和词表有关，所以很多模型会共享这两套权重，这就叫 weight tying。

**展开回答：**

如果 embedding 矩阵记作：

$$
E \in \mathbb{R}^{V \times d_{model}}
$$

输出投影通常是：

$$
W_{out} \in \mathbb{R}^{d_{model} \times V}
$$

weight tying 就是令：

$$
W_{out} = E^T
$$

这样可以减少参数，并让输入和输出使用更一致的词表表示空间。

### 19）为什么 attention 更适合长距离依赖，但长文本还是会遇到瓶颈？

**面试短答：**

因为“依赖路径短”和“计算便宜”是两回事。attention 在关系建模上很直接，任意两位置一跳就能交互；但标准 self-attention 需要处理所有两两位置交互，所以时间和显存通常都是二次增长。

**展开回答：**

如果序列长度为 `L`，attention score 的 shape 是：

$$
L \times L
$$

这意味着长文本里最贵的是 attention matrix 本身，而不是 FFN。因此现代改进经常围绕长序列效率展开，比如稀疏 attention、线性 attention、FlashAttention。

### 20）Attention 是不是替代了全部建模能力？

**面试短答：**

不是。Transformer 的表达能力来自 attention + FFN + 残差 + 归一化 + 位置建模 的组合。只讲 attention 而忽略其它模块，说明理解还不完整。

**展开回答：**

attention 负责 token 间的信息交互，FFN 负责逐位置非线性变换，残差和归一化负责可训练性，位置编码负责顺序信息。缺任何一块，标准 Transformer 的能力都会明显下降。